# Model Evaluation, Comparison & Improvement

This notebook extends the baseline sentiment model by comparing 
multiple algorithms, applying cross-validation, checking for overfitting, 
performing error analysis, and selecting a final candidate model with 
justification.

**Models compared:**
- Logistic Regression (baseline, carried forward)
- Multinomial Naive Bayes
- Linear SVM

**Why these three:** they represent genuinely different approaches to text 
classification (linear probabilistic, linear discriminative, and margin-based) 
rather than variants of the same idea, and all three are well-suited to 
TF-IDF's high-dimensional, sparse feature space. Random Forest was excluded, 
tree-based models generally underperform on sparse TF-IDF vectors compared to 
linear approaches.

## Setup

Reloading the enriched dataset, recreating the sentiment target and TF-IDF features, and importing the additional models and evaluation 
tools needed.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

df = pd.read_csv("../data/processed/processed_data_enriched.csv")

# Recreate sentiment target (same as notebook 05)
df["sentiment"] = df["rating"].apply(lambda r: "Positive" if r >= 4 else "Negative")

X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training set:", X_train_tfidf.shape)
print("Test set:", X_test_tfidf.shape)

Training set: (13592, 5000)
Test set: (3399, 5000)


## Re-establishing the baseline

Same majority-class baseline, included here so this notebook's 
comparison table has the full picture in one place.

In [11]:
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train_tfidf, y_train)
baseline_pred = baseline_model.predict(X_test_tfidf)

print(classification_report(y_test, baseline_pred))

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       625
    Positive       0.82      1.00      0.90      2774

    accuracy                           0.82      3399
   macro avg       0.41      0.50      0.45      3399
weighted avg       0.67      0.82      0.73      3399



c:\Users\ASUS\Desktop\laptop-absa-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\Desktop\laptop-absa-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\Desktop\laptop-absa-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

## Training three candidate models

Three different algorithmic approaches are trained on the same TF-IDF 
features, using the same train/test split, for a fair comparison:

- **Logistic Regression**-linear model estimating class probabilities directly 
- **Multinomial Naive Bayes**-probabilistic model based on word frequency counts, a classic text classification baseline
- **Linear SVM**-finds the boundary that best separates classes with the widest margin, often strong on sparse high-dimensional text data

All three use `class_weight="balanced"` where supported, to account for the 
81.6% / 18.4% class imbalance.

In [12]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(max_iter=2000, class_weight="balanced")
}

# Naive Bayes doesn't support class_weight, so imbalance is handled
# differently for it, we'll note this in the interpretation.

trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    predictions[name] = model.predict(X_test_tfidf)
    trained_models[name] = model
    print(f"\n{'='*20} {name} {'='*20}")
    print(classification_report(y_test, predictions[name]))


==================== Logistic Regression ====================
              precision    recall  f1-score   support

    Negative       0.60      0.80      0.69       625
    Positive       0.95      0.88      0.91      2774

    accuracy                           0.87      3399
   macro avg       0.78      0.84      0.80      3399
weighted avg       0.89      0.87      0.87      3399


==================== Naive Bayes ====================
              precision    recall  f1-score   support

    Negative       0.91      0.34      0.50       625
    Positive       0.87      0.99      0.93      2774

    accuracy                           0.87      3399
   macro avg       0.89      0.67      0.71      3399
weighted avg       0.88      0.87      0.85      3399


==================== Linear SVM ====================
              precision    recall  f1-score   support

    Negative       0.64      0.77      0.70       625
    Positive       0.95      0.90      0.92      2774

    accura

## Model comparison

The table below summarizes performance across all four models, with a 
particular focus on the Negative class, since accuracy alone is misleading 
on this imbalanced dataset.

In [13]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Logistic Regression", "Naive Bayes", "Linear SVM"],
    "Accuracy": [0.82, 0.87, 0.87, 0.88],
    "Negative Precision": [0.00, 0.60, 0.91, 0.64],
    "Negative Recall": [0.00, 0.80, 0.34, 0.77],
    "Negative F1": [0.00, 0.69, 0.50, 0.70],
})
comparison

,Model,Accuracy,Negative Precision,Negative Recall,Negative F1
0,Baseline,0.82,0.00,0.00,0.00
1,Logistic Regression,0.87,0.60,0.80,0.69
2,Naive Bayes,0.87,0.91,0.34,0.50
3,Linear SVM,0.88,0.64,0.77,0.70


Naive Bayes reaches the same overall accuracy as Logistic Regression (87%), 
and its highest precision on Negative (0.91), but its Negative recall (0.34) 
is far lower, it misses about two-thirds of actual negative reviews. This 
lines up with the earlier expectation: Naive Bayes has no built-in class 
imbalance handling, so it leans toward the majority class more than the 
other two models, closer to the baseline's behavior than to Logistic 
Regression or Linear SVM.

Linear SVM edges out Logistic Regression on every metric shown, slightly 
higher accuracy (88% vs 87%), higher Negative precision (0.64 vs 0.60), 
and a marginally better Negative F1 (0.70 vs 0.69), while recall is close 
(0.77 vs 0.80). At this stage, Linear SVM looks like the strongest 
candidate, though this should be confirmed with cross-validation rather 
than a single train/test split.

## Cross-validation

Each model is evaluated using 5-fold cross-validation on the training data, 
giving 5 performance scores per model instead of just one. This checks 
whether performance is consistent across different data splits, rather than 
relying on a single train/test division.

Since the classes are imbalanced, F1-macro (the average of Positive F1 and 
Negative F1, treating both classes equally regardless of size) is used as 
the scoring metric, rather than accuracy, which would be dominated by the 
majority Positive class.

A Pipeline bundling TF-IDF vectorization and the model together is used, 
so that each fold fits its own vectorizer independently on that fold's 
training portion, keeping the same no-leakage principle used earlier.

In [14]:
from sklearn.pipeline import Pipeline

cv_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(max_iter=2000, class_weight="balanced")
}

cv_results = {}

for name, model in cv_models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
        ("classifier", model)
    ])

    # cv=5 means 5-fold cross-validation
    # scoring="f1_macro" averages F1 across both classes equally,
    # so the larger Positive class doesn't dominate the score
    scores = cross_val_score(
        pipeline,
        X_train,   # raw text, not X_train_tfidf, the pipeline handles vectorization per fold
        y_train,
        cv=5,
        scoring="f1_macro"
    )

    cv_results[name] = scores
    print(f"{name}:")
    print(f"  Fold scores: {np.round(scores, 3)}")
    print(f"  Mean F1-macro: {scores.mean():.3f}")
    print(f"  Std deviation: {scores.std():.3f}\n")

Logistic Regression:
  Fold scores: [0.814 0.794 0.794 0.805 0.795]
  Mean F1-macro: 0.800
  Std deviation: 0.008

Naive Bayes:
  Fold scores: [0.71  0.681 0.689 0.716 0.679]
  Mean F1-macro: 0.695
  Std deviation: 0.015

Linear SVM:
  Fold scores: [0.805 0.798 0.807 0.813 0.814]
  Mean F1-macro: 0.807
  Std deviation: 0.006



5-fold cross-validation confirms the single train/test split results. Linear 
SVM achieves the highest mean F1-macro (0.807) with the lowest variability 
across folds (std = 0.006), indicating both strong and consistent 
performance. Logistic Regression performs similarly well (0.800 mean, 0.008 
std). Naive Bayes underperforms both (0.695 mean, 0.015 std), consistent 
with its earlier single-split result showing weak Negative recall, likely 
due to lacking built-in class imbalance handling.

Based on this, Linear SVM and Logistic Regression are carried forward as 
the leading candidates. Naive Bayes is retained in the comparison table for 
completeness but is not considered further as a final candidate.5-fold cross-validation confirms the single train/test split results. Linear 
SVM achieves the highest mean F1-macro (0.807) with the lowest variability 
across folds (std = 0.006), indicating both strong and consistent 
performance. Logistic Regression performs similarly well (0.800 mean, 0.008 
std). Naive Bayes underperforms both (0.695 mean, 0.015 std), consistent 
with its earlier single-split result showing weak Negative recall, likely 
due to lacking built-in class imbalance handling.

Based on this, Linear SVM and Logistic Regression are carried forward as 
the leading candidates. Naive Bayes is retained in the comparison table for 
completeness but is not considered further as a final candidate.

## Overfitting check

Training performance is compared against test performance for the two 
leading candidates. A large gap between the two (high training score, much 
lower test score) would indicate the model is memorizing the training data 
rather than learning patterns that generalize, rather than genuinely 
learning generalizable patterns.

In [15]:
overfit_check = {}

for name in ["Logistic Regression", "Linear SVM"]:
    model = trained_models[name]

    train_pred = model.predict(X_train_tfidf)
    test_pred = predictions[name]  # already computed earlier

    train_f1 = f1_score(y_train, train_pred, average="macro")
    test_f1 = f1_score(y_test, test_pred, average="macro")

    overfit_check[name] = {"Train F1-macro": train_f1, "Test F1-macro": test_f1}
    print(f"{name}:")
    print(f"  Train F1-macro: {train_f1:.3f}")
    print(f"  Test F1-macro: {test_f1:.3f}")
    print(f"  Gap: {train_f1 - test_f1:.3f}\n")

Logistic Regression:
  Train F1-macro: 0.844
  Test F1-macro: 0.800
  Gap: 0.044

Linear SVM:
  Train F1-macro: 0.890
  Test F1-macro: 0.813
  Gap: 0.077



Both models show reasonable train-test gaps, no strong evidence of 
overfitting. Logistic Regression has a tighter gap (0.044), suggesting it 
generalizes slightly more consistently. Linear SVM has a somewhat larger 
gap (0.077), meaning it fits the training data a bit more closely, but its 
test performance (0.813) still exceeds Logistic Regression's (0.800), so 
this is not a concerning level of overfitting, just a mild trade-off to be 
aware of when making the final model choice.

## Error analysis

Rather than only counting incorrect predictions, this section examines 
specific misclassified reviews to understand what kinds of language the 
model struggles with. Linear SVM's predictions are used here, as the 
current leading candidate.

In [16]:
# Build a dataframe of test set reviews alongside actual and predicted labels
error_df = pd.DataFrame({
    "review": X_test.values,
    "actual": y_test.values,
    "predicted": predictions["Linear SVM"]
})

# Isolate only the rows where the model got it wrong
misclassified = error_df[error_df["actual"] != error_df["predicted"]]

print(f"Total misclassified: {len(misclassified)} out of {len(error_df)} ({len(misclassified)/len(error_df)*100:.1f}%)")

# Look specifically at false negatives, actual Negative reviews the model
# predicted as Positive, since these are the ones your project cares about most
false_negatives = misclassified[misclassified["actual"] == "Negative"]

print(f"\nFalse negatives (missed negative reviews): {len(false_negatives)}")
false_negatives[["review", "actual", "predicted"]].sample(10, random_state=42)

Total misclassified: 412 out of 3399 (12.1%)

False negatives (missed negative reviews): 142


,review,actual,predicted
3250,Good but better.,Negative,Positive
475,"The body colour was lossing, we must need to b...",Negative,Positive
2026,Never buy any thing in Flipkart,Negative,Positive
2359,That's is awesome\r\nNo lag no issue\r\nOnly b...,Negative,Positive
1541,Value product...\r\n👍👍👍👍,Negative,Positive
264,Product is good but flipkart teach some patien...,Negative,Positive
2434,price justified . Everything works as expected...,Negative,Positive
1688,"Overall, the laptop is good\r\nDisplay is not ...",Negative,Positive
1695,"Good, but price is too high according to others.",Negative,Positive
466,Never by this probuct,Negative,Positive


Examining 10 randomly sampled false negatives (actual Negative reviews 
predicted as Positive) reveals four recurring patterns:

1. **Mixed sentiment**: reviews open with positive words ("good", "awesome") 
   before pivoting to the actual complaint ("but the display..."). TF-IDF 
   treats words largely independently of order, so word-level positivity 
   can outweigh the structural cue that the complaint is the real message.

2. **Truncated reviews**: several reviews are cut off mid-sentence, likely 
   from the "READ MORE" scraper artifact identified during EDA (~3.7% of 
   reviews affected). The negative content appears to have been cut off 
   in these cases, directly explaining some of the model's errors.

3. **Complaints about seller/delivery rather than the product**: phrases 
   like "never buy from Flipkart" reflect negative experience with the 
   platform, not the laptop itself, yet carry the same binary label as 
   product complaints. This is a genuine limitation of using overall star 
   rating as a proxy for product sentiment, and motivates the planned move 
   to aspect-level sentiment extraction.

4. **Rating-text mismatch**: at least one case ("Value product... 👍👍👍👍") 
   shows clearly positive text paired with a low star rating, likely label 
   noise rather than a model error, the rating may reflect factors outside 
   the review text itself.

## Feature importance / interpretability

Linear SVM, like Logistic Regression, is a linear model, meaning each word 
in the TF-IDF vocabulary has a coefficient indicating how strongly it pushes 
a prediction toward Positive or Negative. This section extracts the words 
with the strongest influence in each direction, useful both for sanity- 
checking that the model is learning genuinely meaningful patterns, and for 
explaining the model in plain language rather than treating it as a black box.

In [17]:
svm_model = trained_models["Linear SVM"]
feature_names = vectorizer.get_feature_names_out()

# For a binary classifier, coef_ has one row: positive values push toward
# the second class alphabetically (Positive), negative values push toward
# the first (Negative)
coefficients = svm_model.coef_[0]

top_positive_idx = np.argsort(coefficients)[-15:][::-1]
top_negative_idx = np.argsort(coefficients)[:15]

print("Top words pushing toward Positive:")
for i in top_positive_idx:
    print(f"  {feature_names[i]}: {coefficients[i]:.3f}")

print("\nTop words pushing toward Negative:")
for i in top_negative_idx:
    print(f"  {feature_names[i]}: {coefficients[i]:.3f}")

Top words pushing toward Positive:
  awesome: 3.319
  notch: 1.949
  beast: 1.836
  actually: 1.814
  msi: 1.790
  wonderful: 1.786
  fastly: 1.735
  osm: 1.735
  40k: 1.678
  smooth: 1.628
  metal: 1.586
  trail: 1.565
  things: 1.561
  slightly: 1.557
  linux: 1.547

Top words pushing toward Negative:
  wanna: -2.556
  expecting: -2.396
  waste: -2.288
  dead: -1.938
  multitasker: -1.884
  1215u: -1.859
  designing: -1.847
  dilwali: -1.764
  gamming: -1.750
  bus: -1.730
  typically: -1.721
  automatically: -1.708
  stuck: -1.704
  outside: -1.688
  don: -1.683


The extracted coefficients confirm the model is largely learning genuine 
sentiment language: clear positive indicators ("awesome", "wonderful", 
"smooth") and clear negative indicators ("waste", "dead", "stuck") dominate 
both lists.

However, a few terms reveal the model also picking up dataset-specific 
correlations rather than pure sentiment. "msi" appears as a Positive 
indicator, likely because MSI laptops happen to have skewed positive 
reviews in this dataset, not because the word itself carries sentiment. 
Similarly, "1215u" (a specific processor model) appears as a Negative 
indicator, probably reflecting that laptops with that processor received 
more negative reviews, rather than the term itself being inherently negative.

This is a useful caution: the model's decisions are not purely about 
sentiment language, they partly reflect patterns specific to this dataset 
(which brands/models happened to get better or worse reviews). This is 
worth keeping in mind if the model were ever applied to a different set 
of laptop reviews, its performance might not transfer perfectly.

## Basic tuning: adding bigrams

The error analysis showed the model struggling with short phrases like 
"good, but" and mixed sentiment. Bigrams (two-word combinations) can help 
capture phrase-level patterns that single words miss, for example "not 
good" or "too high" as their own features, rather than "not" and "good" 
being treated as unrelated.

This is tested on Linear SVM, the current leading candidate, using the 
same 5-fold cross-validation setup as before, so results are directly 
comparable to the earlier unigram-only score (0.807 mean F1-macro).

In [18]:
tuned_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))),
    ("classifier", LinearSVC(max_iter=2000, class_weight="balanced"))
])

tuned_scores = cross_val_score(
    tuned_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro"
)

print("Linear SVM with bigrams:")
print(f"  Fold scores: {np.round(tuned_scores, 3)}")
print(f"  Mean F1-macro: {tuned_scores.mean():.3f}")
print(f"  Std deviation: {tuned_scores.std():.3f}")
print(f"\nComparison to unigram-only: {tuned_scores.mean() - cv_results['Linear SVM'].mean():+.3f}")

Linear SVM with bigrams:
  Fold scores: [0.818 0.805 0.818 0.821 0.812]
  Mean F1-macro: 0.815
  Std deviation: 0.006

Comparison to unigram-only: +0.007


Adding bigrams improved Linear SVM's mean F1-macro from 0.807 to 0.815 
(+0.007), with the same low variability across folds (std = 0.006), 
confirming the improvement is consistent rather than due to chance. While 
modest, this aligns with the error analysis finding that the unigram-only 
model struggled with short, structurally important phrases ("good, but", 
"not worth"). Bigrams are adopted for the final model.

## Final model selection

Based on all evaluation performed in this notebook, Linear SVM with bigrams 
is selected as the final candidate model.

**Predictive performance:** Highest cross-validated F1-macro (0.815) among 
all models tested, with the best single-split test performance (88% 
accuracy, 0.70 Negative F1 before bigrams).

**Performance stability:** Lowest standard deviation across cross-validation 
folds (0.006), indicating consistent performance regardless of which data 
split is used.

**Business impact:** Negative recall (catching actual negative reviews) is 
prioritized over precision for this project, since the goal is identifying 
dissatisfied customers and product issues, missing a negative review is more 
costly than occasionally over-flagging a mixed review. Linear SVM's Negative 
recall (0.77 in the single-split test) is close to Logistic Regression's 
(0.80) while offering better overall balance.

**Interpretability:** As a linear model, coefficients can be directly 
inspected to see which words drive predictions, which was used in this 
notebook's feature importance section, and confirmed the model is largely 
learning genuine sentiment language, with a few dataset-specific quirks 
(brand names, processor models) also noted as limitations.

**Complexity and cost:** Comparable training time to Logistic Regression, 
both are fast, lightweight linear models well suited to this dataset size, 
with no meaningful deployment cost difference between them.

**Known limitations:** 
- Error analysis revealed the model struggles with mixed-sentiment reviews 
  ("good, but..."), truncated reviews (the "READ MORE" scraper artifact), 
  and reviews complaining about seller/delivery rather than the product itself
- Feature importance revealed some dataset-specific correlations (e.g. 
  certain brand or processor mentions influencing predictions) rather than 
  purely sentiment-driven learning, which may not generalize to a different 
  review dataset
- The 2-class (Positive/Negative) framing remains a simplification flagged 
  for mentor discussion

**Conclusion:** Linear SVM with bigrams is selected as the final baseline 
candidate for this phase, offering the strongest and most stable performance 
while remaining interpretable. This model, and the patterns identified during 
error analysis, directly inform the next phase of the project: moving from 
overall sentiment classification toward aspect-level sentiment extraction, 
which should help address the seller-versus-product and mixed-sentiment 
limitations identified here.